# Papapanagiotou Panagiotis

#### Δίνεται η παρακάτω συνάρτηση:
```Python
def process_numbers(nums):
    result = []
    for n in nums:
        if n % 2 == 0:
            result.append(n * 2)
        else:
            result.append(n + 3)
    return sum(result)

nums = [1, 2, 3, 4]
```




### **Μέρος Α - Χωρίς Chain of Thought**  
- Γράψτε ένα prompt που ζητά απευθείας το αποτέλεσμα της συνάρτησης. 

### **Μέρος Β – Με Chain of Thought**
- ### Μετατρέψτε το prompt ώστε:
    - να ζητά από το μοντέλο να σκεφτεί **βήμα-βήμα**
    - να εξηγεί τι γίνεται σε κάθε επανάληψη του loop
    - να καταλήγει στο τελικό αποτέλεσμα

#### My basic setup code

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
 
load_dotenv(Path.cwd().parent / "../module_02_prompt_enginnering/.env")

from anthropic import Anthropic

claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = "claude-sonnet-4-6" 

if claude_client:
    print(f"Antropic client ready - using model {CLAUDE_MODEL}")

Antropic client ready - using model claude-sonnet-4-6


#### Η συνάρτηση που καλεί το API της Athropic

In [2]:
def ask_anthropic(prompt: str, system: str | None = None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model: str= "claude-sonnet-4-6", output_format: str | None = None) -> None:
    """
    Send a prompt to Anthropic using the streaming Messages API and return
    the complete generated text.

    Args:
        prompt (str):
            The user prompt to send to the model.
        system (str | None, optional):
            Optional system instruction for the model. In the streaming API
            for this SDK version, it is converted into the required list format.
            Defaults to None.
        temperature (float, optional):
            Controls randomness of the response. Defaults to 0.7.
        max_resp_tokens (int, optional):
            Maximum number of tokens in the generated response. Defaults to 800.
        ai_model (str, optional):
            The model name to use. Defaults to "claude-sonnet-4-6".
        output_format (str | None, optional):
            Optional output format for the streaming API, if supported by the
            installed SDK version. Defaults to None.

    Returns: None
    """

    params = {
        "model": ai_model,
        "max_tokens" : max_resp_tokens,
        "temperature": temperature,
        "messages": [
            {'role': 'user', 'content': prompt}
        ],
    }

    if system:
        params["system"] = [{"type": "text", "text": system}]

    if output_format:
        params["output_format"] = output_format
    
    with claude_client.messages.stream(**params) as stream:
        full_text = ""
        for text in stream.text_stream:
            full_text += text
            clear_output(wait=True)
            display(Markdown(full_text))
    
    #return full_text

display(Markdown(f"**The function has run**"))

**The function has run**

### **Απάντηση – Χωρίς Chain of Thought**

In [18]:
prompt = """We have this function, calculate ONLY the number that will return:
```Python
def process_numbers(nums):
    result = []
    for n in nums:
        if n % 2 == 0:
            result.append(n * 2)
        else:
            result.append(n + 3)
    return sum(result)
```
As input we have: nums = [1, 2, 3, 4]

Return ONLY the number.
"""
ask_anthropic(prompt)

Let me trace through:

- 1 (odd): 1 + 3 = 4
- 2 (even): 2 * 2 = 4
- 3 (odd): 3 + 3 = 6
- 4 (even): 4 * 2 = 8

sum = 4 + 4 + 6 + 8 = **22**

### Το έτρεξα με πάρα πολλές παραλλαγές και δεν κατάφερα να πάρω ώς αποτέλεσμα μόνο το νούμερο του αποτελέσματος.
#### Κάποιες απο τις παραλλαγές έτσι για την ιστορία της υπόθεσης ήταν τα παρακάτω:
- #### Στο κάτω μέρος του prompting:
    - Please your response MUST be Just the number. 
    - What will return?(Only the number)
    - What is the calculation of this function? Answer JUST the number! 
    - Your Answer MUST BE JUST the NUMBER!
- #### Στην αρχή του prompting:
    - We have this function, calculate what number will this return
    - We have this function, calculate ONLY the number that will return


### **Απάντηση – Με Chain of Thought**

In [19]:
prompt = """We have this function, what will return?:
```Python
def process_numbers(nums):
    result = []
    for n in nums:
        if n % 2 == 0:
            result.append(n * 2)
        else:
            result.append(n + 3)
    return sum(result)
```
As input we have: nums = [1, 2, 3, 4]

Lets trace through the code before your answer.
"""
ask_anthropic(prompt)

## Tracing Through the Code

**Input:** `nums = [1, 2, 3, 4]`

`result` starts as an empty list `[]`

---

### Loop Iterations:

| `n` | `n % 2 == 0`? | Operation | Value Added | `result` |
|-----|--------------|-----------|-------------|----------|
| 1 | 1 % 2 = 1 → **False** | 1 + 3 | 4 | [4] |
| 2 | 2 % 2 = 0 → **True** | 2 * 2 | 4 | [4, 4] |
| 3 | 3 % 2 = 1 → **False** | 3 + 3 | 6 | [4, 4, 6] |
| 4 | 4 % 2 = 0 → **True** | 4 * 2 | 8 | [4, 4, 6, 8] |

---

### Final Step:

```python
return sum([4, 4, 6, 8])
```
`4 + 4 + 6 + 8 = 22`

---

## ✅ The function returns **22**

## Ποια είναι η διαφορά στις δύο απαντήσεις;

### Επειδή δεν βρήκα κάποια ουσιαστική διαφορά στις δύο απαντήσεις, ούτε στη ακρίβεια του αποτελέσματος(δεν έκανε ούτε μια φορά λάθος σε πάνω απο 20 διαφορετικές προσπάθειες) θέλω να πειραματιστώ και με το OpenAI LLM. 

# Ουσιαστικά εδώ τελειώνει η Άσκηση για να μην κουράσω. 
#### Η συνέχεια είναι για να ικανοποιήσω την περιέργια μου...

### Basic OpenAI setup

In [20]:
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [21]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

### **Απάντηση – Χωρίς Chain of Thought**

In [22]:
prompt = """We have this function, calculate ONLY the number that will return:
```Python
def process_numbers(nums):
    result = []
    for n in nums:
        if n % 2 == 0:
            result.append(n * 2)
        else:
            result.append(n + 3)
    return sum(result)
```
As input we have: nums = [1, 2, 3, 4]

Remember please to write ONLY the number.
"""
result = ask_open_ai(prompt, temperature=0.0)
display(Markdown(result))

20

### Ωπ! Τι είχαμε εδώ με την πρώτη προσπάθεια κιόλας; 🤔

### **Απάντηση – Με Chain of Thought**

In [23]:
prompt_cot = """We have this function, what will return?:
```Python
def process_numbers(nums):
    result = []
    for n in nums:
        if n % 2 == 0:
            result.append(n * 2)
        else:
            result.append(n + 3)
    return sum(result)
```
As input we have: nums = [1, 2, 3, 4]

Lets trace through the code before your answer.
"""
cot_result = ask_open_ai(prompt_cot, temperature= 0.0)
display(Markdown(cot_result)) 

Let's trace through the code step by step with the input `nums = [1, 2, 3, 4]`.

1. **Initialization**: 
   - We start with an empty list `result = []`.

2. **First iteration (n = 1)**:
   - Check if `1 % 2 == 0`: This is `False` (1 is odd).
   - Since it's odd, we append `1 + 3` to `result`.
   - `result` becomes `[4]`.

3. **Second iteration (n = 2)**:
   - Check if `2 % 2 == 0`: This is `True` (2 is even).
   - Since it's even, we append `2 * 2` to `result`.
   - `result` becomes `[4, 4]`.

4. **Third iteration (n = 3)**:
   - Check if `3 % 2 == 0`: This is `False` (3 is odd).
   - Since it's odd, we append `3 + 3` to `result`.
   - `result` becomes `[4, 4, 6]`.

5. **Fourth iteration (n = 4)**:
   - Check if `4 % 2 == 0`: This is `True` (4 is even).
   - Since it's even, we append `4 * 2` to `result`.
   - `result` becomes `[4, 4, 6, 8]`.

6. **Final step**:
   - Now we return the sum of the `result` list.
   - The sum is `4 + 4 + 6 + 8 = 22`.

So, the final output of the function `process_numbers(nums)` when called with `nums = [1, 2, 3, 4]` will be **22**.

### Στο OpenAI model είδαμε τεράστια διαφορά και στο αποτέλεσμα και στην μορφή της απάντησης! (αυτό που περιμέναμε δηλαδή). 
#### Το OpenAI προσωπικά με κάνει να νιώθω μεγαλύτερή αυτοπεποίθηση στην χρήση του γιατί παρώτι θεωρητικά καταφέραμε να το "ξεγελάσουμε" παράλληλα με κάνει να αισθάνομαι ότι ανταποκρίνετε σύμφωνα με τις προσδοκίες μας.   